In [ ]:
# ---
# jupyter:
#   jupytext:
#     text_representation:
#       extension: .py
#       format_name: percent
#       format_version: '1.3'
#       jupytext_version: 1.19.6
#   kernelspec:
#     display_name: jaspy
#     language: python
#     name: jaspy
# ---

# NB 04 Fold x seed batch: parametric, Acar and neural models on four purged folds

One task per (fold, seed, network): 4 folds x 3 seeds x 12 networks = 144 trainings (unconditional CODINE, two regime CODINEs, 3 C-CODINE, 6 C-CopFlow), plus the parametric fits (7 unconditional + 14 regime fits per fold, 49 pairs by indexing), the 7 Acar families and the two nonparametric estimators (Gijbels et al., 2011; the empirical checkerboard Bernstein copula of Lu and Ghosh, 2024). Together with the two CODINE variants (unconditional, split) and the 9 conditional networks this gives the 76 candidates of the pooled comparison.

**Acar:** The smoothing bandwidth is selected separately for each (outer fold, family) by purged inner cross-validation using only that outer fold's training observations. A single, reproducibly randomly chosen evaluation seed selects up to 100 validation observations per inner split; Acar is deterministic conditional on those observations and has no neural training seeds. The chosen bandwidths are then used in the corresponding outer fold only. Neural weight decay, batch size and early-stopping settings are pre-specified (not tuned across outer folds); each network uses a purged validation split within its own outer training fold to choose the stopping checkpoint. The kernel-conditional PIT bandwidth was already selected separately for each fold in notebook 00.

Everything stored per job is small (per-step test loss, e1/e2, joint-exceedance probabilities, diagnostics). Results are written one file per job, so an interrupted run resumes.

**Run order:** 00 -> 01 -> 02 -> 03 -> 04 -> 05 -> 06. Notebooks 01 and 02 already compute the parametric fits and the unconditional and regime CODINEs; this notebook resumes them from the shared files in `artifacts/fold_batch` (identical jobs), so nothing is trained twice. The synthetic study (notebooks AAA01 to AAA10) is independent of this notebook.

## At a glance

* **Purpose:** run the 144 neural trainings (4 folds x 3 seeds x 12 nets), the parametric, Acar and nonparametric fits, assemble the loss store and the marginal-uniformity gate, add the Copula Energy Score, the covariate-effect test (GLRT) and the self-consistency check of the networks.
* **Inputs:** `preprocessing_fold_A1/A2/B1/B2` (written by NB00) and `margin_penalty_calibrated.json` (written by NB03). **Outputs:** the artifacts `fold_models`, `fold_nets_seed0`, `ces_scores`, `acar_bandwidth_selection`; one file per fitted network in `fold_batch/states/`; the tables `acar_fits_overview`, `glrt_primary`, `glrt_primary_holm`, `self_consistency_overview` and `wd_sensitivity` of the report, and the numbers behind the text (`neural_ks_pass_rate`, `glrt_real`). Every table is written as LaTeX and CSV.
* **Resuming:** results are written one file per job, so an interrupted run resumes. If a stage is incomplete, run it again (`python run_all.py --stage neural` on JASMIN); worker logs go to `fold_batch/logs/`, tracebacks of failed tasks to `fold_batch/failures/`.
* **Sections:** Fold data, Hyper-parameters, Parametric fits, Acar families, Nonparametric estimators, Neural batch, Training diagnostics, Loss store and gate table, Copula Energy Score, Kendall's tau of the networks, Save, GLRT, Self-consistency, Holm correction.

## Imports and settings

In [ ]:
import os
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"
import warnings; warnings.filterwarnings('ignore')
import hashlib
from pathlib import Path
import numpy as np, pandas as pd

import gate_config as GC
import fold_pipeline as FP
import reporting_tables
reporting_tables.configure("04_continuous_conditional_copulas")

if os.getenv("THESIS_SHARD") and os.getenv("THESIS_STAGE") != "neural":
    raise RuntimeError("Use python run_all.py --stage neural for sharded JASMIN execution. "
                       "Running the full notebook under THESIS_SHARD would stop before neural training.")

from run_support import load_artifact, save_artifact, resolve_max_workers
from run_support import SMOKE, MAX_WORKERS   # production defaults; override with THESIS_SMOKE / THESIS_MAX_WORKERS (see run_support.py)
SEEDS = FP.SEEDS            # (0, 1, 2)
FOLDS = FP.FOLDS
print("smoke:", SMOKE, "| workers:", MAX_WORKERS, "| seeds:", SEEDS, "| folds:", FOLDS)

## Fold data (written by NB00)

A `KeyError` such as `'test_mask'` here means the fold artifacts are stale: re-run the last two cells of `00_preprocessing.ipynb`. `load_fold` checks the keys and says so.

In [ ]:
ARR = FP.load_all_folds()
for f in FOLDS:
    A_ = ARR[f]
    print(f"{f}: train {int(A_['train'].sum())}, test {int(A_['test'].sum())}, b(diagnostic)={A_['b']}, L={A_['purge_final']}, "
          f"high-regime share train {A_['high'][A_['train']].mean():.3f} test {A_['high'][A_['test']].mean():.3f}")
# the validation purge the neural early-stopping splits actually apply (clamped), logged once:
print("val_purge passed explicitly to every neural trainer:", {f: ARR[f]['val_purge'] for f in FOLDS})
n_steps = len(ARR[FOLDS[0]]['u1'])
print("usable steps:", n_steps)

## Hyper-parameters: fold-specific Acar smoothing

Only the Acar bandwidth is tuned: independently for each of the seven families **within each of the four outer training folds**, by purged inner cross-validation. The eight candidate bandwidths are relative to the outer training fold's covariate range. The up-to-100 validation evaluation positions are sampled once per inner split using one reproducibly selected seed shared across families and folds, rather than repeating the search across three neural training seeds (Acar has no such seeds). The weight decay of the nine conditional nets is **not tuned on the real data**: it is one value per group (C-CODINE, C-CopFlow with affine coupling, C-CopFlow with spline coupling) chosen on synthetic data by notebook 03 (`FP.fixed_wd_by_model()`), so it carries no tuning leakage from any real fold. An optional sensitivity table on fold A1 can be switched on with `RUN_WD_SENSITIVITY`; it does not choose anything.

If no bandwidth for a fold and family attains the prespecified 95% validation-coverage threshold, the search **does not silently discard that family**. It selects the candidate with the highest observed coverage (ties: more scored inner folds, then the available mean validation log-likelihood), explicitly records its coverage and flags it as a low-coverage fallback. If no candidate has any finite validation score, the widest tested bandwidth is used as an **unvalidated** emergency default and reported prominently; this is not evidence of a reliable fit. The primary selection rule remains unchanged.

In [ ]:
# Weight decay of the nine conditional nets: one value per group (C-CODINE, C-CopFlow with affine coupling, C-CopFlow with spline coupling),
# chosen by the synthetic calibration of notebook 03 and read from margin_penalty_calibrated.json. No real fold is involved in the choice.
# Optional sensitivity check on the training half of fold A1 (does not choose anything; switched on; set RUN_WD_SENSITIVITY = False to skip).
# The sensitivity jobs belong to the neural stage (and the final assembly, which only reads the resume files); the
# param and acar stages skip them, they only need WD_BY_MODEL and the Acar bandwidth cache set up below.
RUN_WD_SENSITIVITY = os.getenv("THESIS_STAGE", "") in ("", "neural", "assemble")
WD_BY_MODEL = FP.fixed_wd_by_model()
print('weight decay per network (calibrated in notebook 03):', WD_BY_MODEL)
if RUN_WD_SENSITIVITY:
    wd_jobs = FP.build_wd_jobs(ARR, fold=FOLDS[0], smoke=SMOKE)
    _wd = FP.run_batch(wd_jobs, max_workers=MAX_WORKERS, batch_dir=FP.BATCH_DIR / 'wd', fn=FP.wd_tuning_job,
                       key_fn=FP.wd_key, desc='weight-decay sensitivity')
    _, WD_TABLE = FP.select_wd(_wd)
    display(WD_TABLE.pivot_table(index='model', columns='wd', values='best_val_cvpe', sort=False))
    reporting_tables.wd_sensitivity_table(WD_TABLE, used=reporting_tables.wd_used_sentence(FP.WD_CCODINE, FP.WD_CCOPFLOW_AFFINE, FP.WD_CCOPFLOW_SPLINE))

# Acar does NOT have model-training seeds. The sampled inner-CV evaluation points do.
# Draw one member of the existing seed set with a fixed RNG; the same 100 evaluation points per inner block are used
# for all families and bandwidths, which keeps the comparison fair and the selection cheap.
ACAR_EVAL_SEED = int(np.random.default_rng(20261014).choice(SEEDS))
# Refuse a cached tuning result if the outer-fold training data, purge length,
# evaluation seed or numerical tuning implementation changed since selection.
_acar_impl = hashlib.sha256((Path('conditional_copula.py').read_bytes()
                            + Path('data_split.py').read_bytes())).digest()
ACAR_TUNING_INPUT_HASH = {}
for f in FOLDS:
    _hash = hashlib.sha256(_acar_impl)
    A = ARR[f]
    _hash.update(str((ACAR_EVAL_SEED, bool(SMOKE), 'fallback-v1', int(A['purge_final']),
                      FP.ACAR_FAMILIES, 3, 100, 1e-3, 50)).encode())
    _train = np.asarray(A['train'], bool)
    for _name in ('u1', 'u2', 'w'):
        _arr = np.ascontiguousarray(np.asarray(A[_name])[_train], dtype=np.float64)
        _hash.update(str(_arr.shape).encode())
        _hash.update(_arr.tobytes())
    ACAR_TUNING_INPUT_HASH[f] = _hash.hexdigest()
_cached_acar = load_artifact("acar_bandwidth_selection")
_schema_ok = (_cached_acar.get('selection_schema') == 'fold-family-v4'
              and tuple(_cached_acar.get('folds', ())) == tuple(FOLDS)
              and tuple(_cached_acar.get('families', ())) == tuple(FP.ACAR_FAMILIES)
              and _cached_acar.get('tuning_input_hashes') == ACAR_TUNING_INPUT_HASH
              and int(_cached_acar.get('eval_seed', -1)) == ACAR_EVAL_SEED
              and bool(_cached_acar.get('smoke', False)) == bool(SMOKE))
if not _schema_ok and _cached_acar:
    print('Previous Acar bandwidth cache uses old settings/schema; re-tuning separately by outer fold.')
ACAR_H = {f: dict(_cached_acar['bandwidths'].get(f, {})) for f in FOLDS} if _schema_ok else {f: {} for f in FOLDS}
ACAR_SELECTIONS = {f: dict(_cached_acar.get('selections', {}).get(f, {})) for f in FOLDS} if _schema_ok else {f: {} for f in FOLDS}
print('Acar evaluation-subsampling seed:', ACAR_EVAL_SEED)
if any(ACAR_H[f] for f in FOLDS):
    print('Loaded fold-specific cached Acar bandwidths:', ACAR_H)

In [ ]:
# Tune all seven families SEPARATELY on the training rows of EACH outer fold.
# Never use the test rows of a different outer fold to choose its bandwidth.
from conditional_copula import select_model, choose_acar_bandwidths
for outer_fold in FOLDS:
    missing = [fam for fam in FP.ACAR_FAMILIES if fam not in ACAR_H[outer_fold]]
    if not missing:
        continue
    A = ARR[outer_fold]
    tm = np.asarray(A['train'], bool)
    u1_t, u2_t = np.asarray(A['u1'])[tm], np.asarray(A['u2'])[tm]
    w_t = np.asarray(A['w'], float)[tm]
    h_sat = float(np.max(np.abs(w_t)))
    bands = np.linspace(0.5, 1.1 * h_sat, 3 if SMOKE else 8).round(2)
    if len(np.unique(bands)) != len(bands):
        raise RuntimeError(f'Duplicate Acar bandwidth grid values for {outer_fold}: {bands}')
    df_h, _, _ = select_model(u1_t, u2_t, w_t, families=tuple(missing), bandwidths=bands,
                              purge=int(A['purge_final']), n_splits=3,
                              n_jobs=resolve_max_workers(n_tasks=len(missing) * len(bands)),
                              bfgs_options={'gtol': 1e-3, 'maxiter': 50},
                              max_eval_points=100, eval_seed=ACAR_EVAL_SEED)
    choices = choose_acar_bandwidths(df_h, missing, min_coverage=0.95)
    for fam, choice in choices.items():
        ACAR_H[outer_fold][fam] = choice['bandwidth']
        ACAR_SELECTIONS[outer_fold][fam] = choice
        if choice['selection_status'] != 'primary':
            print(f"WARNING Acar {outer_fold}/{fam}: {choice['selection_status']} "
                  f"h={choice['bandwidth']:.3g}, coverage={choice['coverage']:.1%}; "
                  f"{choice['selection_reason']}")
    # Save immediately per completed fold: an interrupted run resumes from the last complete fold.
    save_artifact('acar_bandwidth_selection', bandwidths=ACAR_H, selections=ACAR_SELECTIONS,
                  selection_schema='fold-family-v4', folds=list(FOLDS),
                  families=list(FP.ACAR_FAMILIES), eval_seed=ACAR_EVAL_SEED,
                  tuning_input_hashes=ACAR_TUNING_INPUT_HASH,
                  min_validation_coverage=0.95, smoke=SMOKE)
for f in FOLDS:
    if set(ACAR_H[f]) != set(FP.ACAR_FAMILIES):
        raise RuntimeError(f'Missing Acar bandwidths on {f}: {set(FP.ACAR_FAMILIES) - set(ACAR_H[f])}')
print('Acar selected bandwidths (fold -> family -> h):', ACAR_H)
print('Acar bandwidth choices: primary versus fallback:')
print(pd.DataFrame([dict(outer_fold=f, **ACAR_SELECTIONS[f][fam]) for f in FOLDS for fam in FP.ACAR_FAMILIES]).to_string(index=False))

## Parametric fits per fold (with the ablation without conditional marginals)

Per fold: 7 unconditional fits, 7 per regime (14), and the 49 low/high pairs by indexing the regime losses. the marginal-uniformity KS distance D_max (1000 independent draws, 7 covariate percentiles for conditional models) is computed for the kernel-conditional PIT run; the unconditional-PIT run is the ablation without conditional marginals and is not gated. The AIC-selected pair uses training AIC only.

In [ ]:
PARAM, PARAM_ABL = FP.run_param_stage(ARR, max_workers=MAX_WORKERS)
for f in FOLDS:
    print(f"{f}: AIC-selected pair (low | high) = {PARAM[f]['selected_pair']}   [{PARAM[f]['runtime']:.0f}s]")

## Acar families per fold

One local-likelihood calibration curve per (fold, family) on that fold's training rows, using that fold and family's inner-CV-selected bandwidth; per-step loss on the test rows; marginal-uniformity KS distance D_max, the maximum over the seven covariate percentiles (thresholds and draw counts are set in `gate_config.py`).

In [ ]:
acar_jobs = FP.build_acar_jobs(ARR, ACAR_H, smoke=SMOKE)
_res = FP.run_batch(acar_jobs, max_workers=MAX_WORKERS, batch_dir=FP.BATCH_DIR / 'acar', fn=FP.acar_job,
                    key_fn=FP.acar_key, desc='Acar folds')
ACAR = {(r['fold'], r['family']): r for r in _res.values()}
reporting_tables.show_table(pd.DataFrame([dict(fold=f, family=fam, h=r['h'], KS_D=r.get('ks', np.nan), runtime_s=round(r['runtime']))
                    for (f, fam), r in sorted(ACAR.items())]), 'acar_fits_overview', caption='Acar fits: bandwidth, marginal KS distance and runtime.', index=False)

## Nonparametric estimators per fold: Gijbels et al. (2011) and the empirical checkerboard Bernstein copula (Lu and Ghosh, 2024)

Both estimators are fitted on each fold's training rows only (Gijbels: local-linear weights with the AMISE plug-in bandwidths of `gijbels_copula`; ECBC: degrees n^alpha averaged over five values of alpha in (1/3, 2/3), see `ecbc_copula`). Each returns the cell masses of the conditional copula, so a test step is scored as in the Copula Energy Score: masses on a grid of 81 covariate values spanning the test range (clipped to the training range) and linear interpolation of each step's score. The per-step loss is the CVPE integrand with the conditional means taken from the cell masses merged to 32 cells per margin and interpolated between the cell centres, so the CVPE of these two rows carries a small discretisation error of its own and is read as a reference level; the Copula Energy Score on the 64 x 64 cells is like-for-like with the other models. The marginal-uniformity distance D_max is computed as for Acar: the maximum over the seven training-range percentiles of the covariate, 14 KS tests, 1,000 draws from the cell masses each. These jobs are deterministic (no seeds).

In [ ]:
np_jobs = FP.build_nonparametric_jobs(ARR, smoke=SMOKE)
_res = FP.run_batch(np_jobs, max_workers=MAX_WORKERS, batch_dir=FP.BATCH_DIR / 'nonparam', fn=FP.nonparametric_job,
                    key_fn=FP.nonparam_key, desc='nonparametric folds')
NONPARAM = {(r['fold'], r['kind']): r for r in _res.values()}
print({f"{f} {k}": ("pass" if r['ks_pass'] else "FAIL") for (f, k), r in sorted(NONPARAM.items())})

## Neural batch: fold x seed x network (144 tasks)

Seeds 0, 1, 2 reach the training, the early-stopping split seed and the normalisation calibration (`calib_seed`). Training settings are the same as in the weight-decay tuning stage (batch 32, validation fraction 0.25, max 8000 epochs, patience 5, validation every 100 epochs); the weight decay is the calibrated value of the network's group (C-CODINE, C-CopFlow affine, C-CopFlow spline coupling; notebook 03). Only fold and seed change. Seed-0 networks are kept for the diagnostics and figures; the other seeds are stored as per-step predictions only.

In [ ]:
neural_jobs = FP.build_fold_jobs(ARR, WD_BY_MODEL, smoke=SMOKE)
print(len(neural_jobs), "tasks")
NEURAL = FP.run_batch(neural_jobs, max_workers=MAX_WORKERS, batch_dir=FP.BATCH_DIR, desc='neural fold x seed')

## Training diagnostics and marginal-uniformity pass rate

In [ ]:
diag = pd.DataFrame([dict(fold=r['fold'], seed=r['seed'], model=r['model'], ks=r['ks'], ks_pass=r['ks_pass'],
                          best_epoch=r.get('best_epoch'), stopped_epoch=r.get('stopped_epoch'),
                          best_val_cvpe=r.get('best_val_cvpe'), runtime_s=round(r['runtime']),
                          ks_d=r.get('ks', np.nan)) for r in NEURAL.values()])
diag = diag.sort_values(['model', 'fold', 'seed']).reset_index(drop=True)
print("tasks:", len(diag), "| total CPU time: %.1f h" % (diag.runtime_s.sum() / 3600))
cond = diag[diag.ks.notna()]
rate = cond.groupby('model').agg(nets=('ks_pass', 'size'), passed=('ks_pass', 'sum'), max_ks=('ks', 'max'))
rate['pass_rate'] = rate.passed / rate.nets
# diagnostic only: the pool gate below uses the maximum over folds and seeds of each model
reporting_tables.show_table(rate, 'neural_ks_pass_rate', caption='Marginal KS pass rate of the conditional networks.', digits=3)

## Loss store and gate table

`assemble_store` collects candidates and averages neural predictions over all available seeds. Only the marginal-uniformity gate controls admission to the pool: D_max must not exceed the null-calibrated threshold of its check (`gate_config.ks_threshold`; 0.07 for the reference check of two KS tests) in every fold, seed and regime. Missing diagnostics fail. The gate table explains exclusions for all 76 candidates.

In [ ]:
STORE = FP.assemble_store(PARAM, ACAR, NEURAL, n=n_steps, nonparam=NONPARAM)
gt = STORE['gate_table']
print("tensor (steps, models, folds):", STORE['tensor'].shape)
assert len(gt) == GC.N_CANDIDATES, (len(gt), GC.N_CANDIDATES)       # 76 = (7+1) + (49+1) + (7+3+6) + 2
print("candidates:", len(gt), "| with a loss in every fold:", len(STORE['models']), "| in pool after gates:", int(gt.in_pool.sum()))
assert gt.groupby('group').size().to_dict() == {'Acar': 7, 'Binary': 50, 'Neural': 9, 'Nonparametric': 2, 'Unconditional': 8}
print(gt.groupby('group').in_pool.agg(candidates='size', in_pool='sum').to_string())

## Copula Energy Score (additional metric; reads stored fits, trains nothing)

CES scores the fitted dependence distribution of each PIT pair (Ziel and Berk 2019; lower is better). It is reported next to CVPE, never instead of it and never pooled with it. Primary target: the raw joint score of the model's own distribution; sensitivity: the same distribution projected to uniform margins separately at every divergence value. Parametric and Acar models are scored from their stored fits and the nonparametric estimators in their own jobs; networks are scored from their stored best-checkpoint weights. Seeds 1-2 of runs made before three-seed persistence have no stored weights and are listed as *not reconstructable*.

In [ ]:
import ces_score as CR
CES_K, CES_G = (32, 21) if SMOKE else (64, 81)
CES_NEURAL = CR.run_ces_jobs(ARR, max_workers=MAX_WORKERS, K=CES_K, G=CES_G)        # resumable, one file per network
CES_PARAM = CR.ces_parametric(PARAM, ARR, FOLDS, K=CES_K)
CES_ACAR = CR.ces_acar(ACAR, ARR, FOLDS, K=CES_K, G=CES_G)
CES_NP = CR.ces_nonparametric(NONPARAM, FOLDS)
_cn, CES_SEEDS, CES_COV = CR.collect_neural(CES_NEURAL)
CES_STORE = CR.assemble_ces_store(STORE, [CES_PARAM, CES_ACAR, CES_NP, _cn], FOLDS, seeds_used=CES_SEEDS, target='raw')
CES_STORE_CAL = CR.assemble_ces_store(STORE, [CES_PARAM, CES_ACAR, CES_NP, _cn], FOLDS, seeds_used=CES_SEEDS, target='calibrated')
_ces_raw = CR.ces_table(CES_STORE, K=CES_K, G=CES_G)
CES_TAB = _ces_raw.join(CR.ces_table(CES_STORE_CAL, K=CES_K, G=CES_G)[['ces']].rename(columns={'ces': 'ces_calibrated'}))
CES_TAB.attrs.update(_ces_raw.attrs)   # join drops attrs when the two tables' attrs differ (pandas 2.3)
print('CES status of the networks:', CES_COV.groupby('status').size().to_dict())
print(f"{len(CES_STORE['models'])} of {len(STORE['models'])} candidates have a CES in every fold; independence reference CES {CES_TAB.attrs['independence_ces']:.5f}")
print("estimator:", CES_TAB.attrs['estimator'])
save_artifact("ces_scores", store_raw=CES_STORE, store_calibrated=CES_STORE_CAL, table=CES_TAB, coverage=CES_COV,
              K=CES_K, G=CES_G, smoke=SMOKE)

## Training stability

In [ ]:
# Stability-only check (pre-declared rule): finite best validation CVPE and stopped before the epoch cap.
# The networks share one recipe (weight decay per model group as set in fold_pipeline: WD_CCODINE, WD_CCOPFLOW_AFFINE, WD_CCOPFLOW_SPLINE; clip 5; validation every 100 epochs). Test CVPE is NOT used here.
STAB = FP.training_stability_table(NEURAL)
if len(STAB):
    print(f"networks that trained stably: {int(STAB.stable.sum())} of {len(STAB)}")
    cod = STAB[STAB.model.str.contains('CODINE') & ~STAB.model.str.startswith('C-')]
    if len(cod) and not cod.stable.all():
        print('WARNING: unstable CODINE network(s) under the shared recipe; report the CODINE comparison as confounded by the recipe.')

## Kendall's tau as a function of w (networks, mean over seeds, per fold)

Read by NB06 (`mcs_survivors`). The Acar curves come from the calibration curves already stored.

In [ ]:
from codine import conditional_calibration_curve
from conditional_copula_flow import flow_calibration_curve
# tau(w) of every network, averaged over ALL seeds (the same seeds that enter the losses, CES and self-consistency tables).
# TAU[f][m] is the seed mean (read by NB06); TAU_SD[f][m] is the SD over seeds; TAU_SEEDS keeps the single curves.
TAU, TAU_SD, TAU_SEEDS = {}, {}, {}
n_w, n_mc = (8, 2000) if SMOKE else (60, 20_000)
for f in FOLDS:
    Af = ARR[f]; wt = Af['w'][Af['train']]
    wg = np.linspace(np.quantile(wt, 0.02), np.quantile(wt, 0.98), n_w)
    TAU[f], TAU_SD[f], TAU_SEEDS[f] = {'w_grid': wg}, {'w_grid': wg}, {}
    for m in FP.NEURAL_MODELS:
        curves = {}
        for s in SEEDS:
            rec = NEURAL.get(FP.net_key(f, s, m))
            if rec is None or rec.get('net') is None:
                continue
            curves[s] = np.asarray(conditional_calibration_curve(rec['net'], wg, n_samples=n_mc, seed=1) if m in FP.CCODINE_VARIANTS
                                   else flow_calibration_curve(rec['net'], wg, n_samples=n_mc, seed=1), float)
        if not curves:
            raise RuntimeError(f"no trained network available for tau(w): fold {f}, model {m}")
        M = np.vstack(list(curves.values()))
        TAU[f][m] = M.mean(axis=0)
        TAU_SD[f][m] = M.std(axis=0, ddof=1) if len(M) > 1 else np.full(M.shape[1], np.nan)
        TAU_SEEDS[f][m] = curves
        if len(curves) < len(SEEDS):
            print(f"WARNING tau(w) {f} / {m}: only seeds {sorted(curves)} available")
print("tau(w) curves (seed means) stored for", list(TAU))

## Save

In [ ]:
# slim copies: nets/curves of seed 0 go to their own artifact so the main artifact stays small
neural_slim = {k: {kk: vv for kk, vv in r.items() if kk not in ('net', 'losses')} for k, r in NEURAL.items()}
nets_seed0 = {k: {'net': r['net'], 'losses': r['losses']} for k, r in NEURAL.items() if r.get('net') is not None}
save_artifact("fold_models",
              param=PARAM, param_ablation_uncond_pit=PARAM_ABL, acar=ACAR, nonparam=NONPARAM, neural=neural_slim,
              store=STORE, diag=diag, wd_by_model=WD_BY_MODEL, acar_h=ACAR_H, acar_eval_seed=ACAR_EVAL_SEED,
              tau_curves=TAU, tau_curves_seed_sd=TAU_SD, tau_curves_seeds=TAU_SEEDS, folds=list(FOLDS), seeds=list(SEEDS), smoke=SMOKE,
              arrays={f: {k: v for k, v in ARR[f].items()} for f in FOLDS})
save_artifact("fold_nets_seed0", nets=nets_seed0)
print("saved fold_models and fold_nets_seed0")

## Covariate-effect test (Acar, Craiu and Yao 2013 GLRT)

Question answered: within one parametric family, does the copula dependence change with the (standardised) divergence? Hypotheses: H0 constant calibration function, H1 arbitrary smooth, local-constant alternative. It does not say whether a neural model beats a simpler one and does not replace CVPE, CES or the MCS.

The test runs on the training rows of EACH of the four outer folds (A1, A2, B1, B2). The GLRT involves no neural network, so there is no seed dimension: for a given fold the statistic is deterministic and only the Monte Carlo nulls use random numbers (fixed seeds).

The primary family of a fold is fixed by a rule, not by looking at GLRT results: the parametric unconditional family with the lowest AIC on that fold's training rows (AIC = 2k - 2 logL, k = 1 or 2 for the Student-t; better rotation branch for the signed families). The choice uses no held-out data. The other six families are exploratory with Holm-adjusted p-values. The i.i.d. asymptotic p-value is printed as a reference only. Formal inference uses a Monte Carlo null with the CV bandwidth re-selected in every replicate (parametric bootstrap with the divergence fixed). If the calibrated p-value is not below 0.05 the verdict is "formal significance not established", which is not evidence of no effect. Caveats: estimated PITs, serial dependence, data-driven bandwidth, overlapping folds, size checked only coarsely by simulation (`acar_glrt.py`). Runtime is roughly 9 s per statistic x B replicates per family and fold, split over the workers (four times the single-fold runtime).

In [ ]:
import acar_glrt as GR
RUN_GLRT = os.getenv('THESIS_RUN_GLRT', '1') == '1'     # set THESIS_RUN_GLRT=0 to skip the (long) Monte Carlo nulls
GLRT_BY_FOLD = {}
for GLRT_FOLD in FOLDS:
    A_g = ARR[GLRT_FOLD]; tr = np.asarray(A_g['train'], bool)
    u1_g, u2_g, w_g = np.asarray(A_g['u1'])[tr], np.asarray(A_g['u2'])[tr], np.asarray(A_g['w'], float)[tr]
    # Primary family by a fixed rule applied before any GLRT output exists: lowest AIC of the unconditional parametric fit on this fold's TRAINING rows.
    prim, _ = GR.aic_primary_family(u1_g, u2_g, FP.ACAR_FAMILIES)
    print(f'fold {GLRT_FOLD}: GLRT primary family (lowest training-row AIC) = {prim}')
    if RUN_GLRT:
        expl = [x for x in FP.ACAR_FAMILIES if x != prim]
        res = GR.run_glrt_real(u1_g, u2_g, w_g, prim, exploratory_families=[] if SMOKE else expl,
                               B=9 if SMOKE else 199, workers=resolve_max_workers(n_tasks=9 if SMOKE else 199))
        res.insert(0, 'fold', GLRT_FOLD)
        GLRT_BY_FOLD[GLRT_FOLD] = res
if GLRT_BY_FOLD:
    GLRT = pd.concat(GLRT_BY_FOLD.values(), ignore_index=True)
    display(GLRT[['fold', 'role', 'family', 'n', 'lam', 'h', 'abs_X_over_h', 'p_asymptotic_reference', 'p_param_boot', 'p_holm_exploratory', 'verdict']].round(4))
    reporting_tables.write_table(GLRT, 'glrt_real', index=False, caption='Covariate-effect test (GLRT) for every family and outer fold: primary and exploratory families.')
    reporting_tables.glrt_primary_table(GLRT)
    print('caveats:', GR.CAVEATS)

## Self-consistency of the networks

Every fitted network density has to integrate to one over the unit square.

In [ ]:
# Self-consistency of every network (all folds and seeds): the fitted density integrates to one over the unit square.
# The empty dict makes run_self_consistency reload each network from its state file in <FP.BATCH_DIR>/states; nothing is retrained.
import model_diagnostics as SCT

SC_LONG, SC_SUMMARY = SCT.run_self_consistency({}, ARR, FOLDS, SEEDS, smoke=SMOKE)
SC_OVERVIEW = SCT.save_self_consistency(SC_LONG, SC_SUMMARY)
print("networks without a stored net object:", int((SC_LONG.status == 'no net').sum()))
print("checks per model:", SC_LONG.groupby('model').size().to_dict())
display(SC_OVERVIEW.round(4))

## Holm correction of the GLRT across the folds

In [ ]:
# Holm correction across the four outer folds of the primary-family GLRT bootstrap p-values (role == "primary", column
# p_param_boot of the GLRT table above), judged at alpha = 0.05 and 0.10.
ALPHAS = (0.05, 0.10)

def holm(p):
    """Holm step-down adjusted p-values.

    Valid under any dependence between the tests (Holm, S. (1979). A simple sequentially rejective multiple test procedure.
    Scandinavian Journal of Statistics, 6(2), 65-70).

    Parameters:
    p (array-like): Raw p-values.

    Returns:
    np.ndarray: Holm-adjusted p-values, in the order of p.
    """
    p = np.asarray(p, float); order = np.argsort(p); adj = np.empty(len(p)); run = 0.0
    for rank, i in enumerate(order):
        run = max(run, min(1.0, (len(p) - rank) * p[i])); adj[i] = run
    return adj

if not GLRT_BY_FOLD:
    raise RuntimeError("The Holm table needs the GLRT results: run the GLRT cell with THESIS_RUN_GLRT=1.")
primary = GLRT[GLRT.role == "primary"].set_index("fold").loc[list(FOLDS)]
raw = primary["p_param_boot"].rename("raw_p")
if not raw.notna().all():
    raise RuntimeError("A primary-family bootstrap p-value is missing (B = 0?).")
B_BOOT = int(primary["B"].iloc[0])

table = pd.DataFrame({"Raw p": raw, "Holm-adjusted p": holm(raw.values)})
for a in ALPHAS:
    table[f"Reject at {int(a*100)}%"] = np.where(table["Holm-adjusted p"] < a, "yes", "no")
print("Rejections after Holm correction across folds:",
      ", ".join(f"{int(a*100)}%: {(table['Holm-adjusted p'] < a).sum()} of {len(FOLDS)}" for a in ALPHAS))
display(table.round(3))
reporting_tables.glrt_holm_table(table, B_BOOT)